# cfd10 — Colab teacher training

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU** (or L4).

Then **Runtime -> Run all**. The only thing to edit is `GDRIVE_DATA` in cell 3 — point it at the Drive folder holding the `raw_v16` CSVs. Cell 5 prints the scorecard to paste back into the chat (and auto-saves it to Drive).

In [ ]:
# Cell 1 - Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - Clone the (public) repo and install deps
from pathlib import Path
REPO_DIR = Path('/content/cfd10')
REPO_URL = 'https://github.com/Sovenski/cfd10.git'
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only -q
# Core deps from pyproject; does NOT touch Colab's GPU torch (torch is a dev-only extra).
!pip install -q -e .
!git rev-parse --short HEAD

In [ ]:
# Cell 3 - Point at your Drive data + confirm GPU
import os, torch
GDRIVE_DATA = '/content/drive/MyDrive/cfd10/data/raw_v16'   # <-- EDIT to your raw_v16 folder in Drive
os.makedirs('/content/cfd10/data', exist_ok=True)
link = '/content/cfd10/data/raw_v16'
if not os.path.exists(link):
    os.symlink(GDRIVE_DATA, link)
assert os.path.isdir(link), f'data not found: {link} -> {GDRIVE_DATA} (fix GDRIVE_DATA)'
n_csv = len([f for f in os.listdir(link) if f.endswith('.csv')])
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-ONLY (switch runtime to GPU!)'
print(f'{n_csv} CSV files visible | device: {gpu}')

In [ ]:
# Cell 4 - Train the TCN teacher (beat-the-GBDT-baseline gate). ~minutes on a T4/L4.
!python pipeline/fit_teacher.py --epochs 30

In [ ]:
# Cell 5 - Print the teacher scorecard (copy this back into the chat) + back it up to Drive
import os, shutil
OUT, DRIVE_OUT = '/content/cfd10/outputs', '/content/drive/MyDrive/cfd10/outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in ('teacher_pooled_scorecard.md', 'baseline_pooled_scorecard.md'):
    if os.path.exists(f'{OUT}/{f}'):
        shutil.copy(f'{OUT}/{f}', f'{DRIVE_OUT}/{f}')
print(open(f'{OUT}/teacher_pooled_scorecard.md', encoding='utf-8').read())

In [ ]:
# Cell 6 (optional) - GBDT pooled baseline + label QA for reference
!python pipeline/fit_pooled.py
print(open('/content/cfd10/outputs/baseline_pooled_scorecard.md', encoding='utf-8').read())